# Phase 2: Data Cleaning & Database Load

This notebook loads the raw, messy datasets (Excel and CSV), cleans them using the utility functions defined in `src/cleaning.py`, saves the clean data as CSV files in `data/processed/`, and populates the SQLite database.

In [ ]:
import os
import sys
import pandas as pd

# Append src folder to path
sys.path.append(os.path.abspath('../src'))

import cleaning
import db_connection

## 1. Initialize Database Schema

In [ ]:
db_connection.init_db()

## 2. Clean and Process Projects Data

In [ ]:
print("Loading raw projects...")
df_projects_raw = pd.read_excel("../data/raw/projects_raw.xlsx")
print(f"Raw projects shape: {df_projects_raw.shape}")

df_projects_clean = cleaning.clean_projects(df_projects_raw)
print(f"Cleaned projects shape: {df_projects_clean.shape}")

df_projects_clean.head()

## 3. Clean and Process Material Vendor Data

In [ ]:
print("Loading raw material vendor...")
df_material_raw = pd.read_csv("../data/raw/material_vendor_raw.csv")
print(f"Raw material vendor shape: {df_material_raw.shape}")

df_material_clean = cleaning.clean_material_vendor(df_material_raw)
print(f"Cleaned material vendor shape: {df_material_clean.shape}")

df_material_clean.head()

## 4. Clean and Process Labour Attendance Data

In [ ]:
print("Loading raw labour attendance...")
df_labour_raw = pd.read_csv("../data/raw/labour_attendance_raw.csv")
print(f"Raw labour attendance shape: {df_labour_raw.shape}")

df_labour_clean = cleaning.clean_labour_attendance(df_labour_raw)
print(f"Cleaned labour attendance shape: {df_labour_clean.shape}")

df_labour_clean.head()

## 5. Save Cleaned Data as CSVs

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

df_projects_clean.to_csv("../data/processed/projects_clean.csv", index=False)
df_material_clean.to_csv("../data/processed/material_vendor_clean.csv", index=False)
df_labour_clean.to_csv("../data/processed/labour_attendance_clean.csv", index=False)

print("Processed CSVs saved successfully in data/processed/")

## 6. Load Cleaned Data into Database

In [ ]:
conn = db_connection.get_connection()

# Insert Projects
df_projects_clean.to_sql("projects", conn, if_exists="append", index=False)
print("Loaded projects into database.")

# Insert Material Vendor
df_material_clean.to_sql("material_vendor", conn, if_exists="append", index=False)
print("Loaded material_vendor into database.")

# Insert Labour Attendance
df_labour_clean.to_sql("labour_attendance", conn, if_exists="append", index=False)
print("Loaded labour_attendance into database.")

# Verify counts
cur = conn.cursor()
for table in ["projects", "material_vendor", "labour_attendance"]:
    cur.execute(f"SELECT count(*) FROM {table}")
    print(f"Table {table}: {cur.fetchone()[0]} rows loaded")

conn.close()
print("Database population completed successfully!")